# Multi-Scale CNN-BiLSTM Autoencoder (MSCNN-BiLSTM-AE) for NIDS

Notebook ini adalah implementasi **SOTA (State-of-the-Art)** untuk deteksi intrusi jaringan menggunakan pendekatan **Unsupervised Learning**.

## Tujuan
Mendeteksi serangan *Zero-Day* pada dataset **CSE-CIC-IDS2018** setelah dilatih HANYA pada trafik normal **CIC-IDS2017**.

## Arsitektur Model
- **Multi-Scale CNN:** Menangkap pola spasial jangka pendek, menengah, dan panjang (Kernel 3, 5, 7).
- **Bi-Directional LSTM:** Menangkap konteks temporal dua arah.
- **Attention Mechanism:** Fokus pada fitur paling signifikan.
- **Autoencoder:** Mendeteksi anomali berdasarkan *reconstruction error*.


## Colab Bootstrap (Colab-only)

Section ini hanya aktif saat runtime Google Colab.
Fungsinya: mount Drive, clone/pull branch target, optional symlink data dari Drive, setup `kaggle.json`,
dan symlink semua output artifact (`data/research`, `models/research`, `results/research`, `reports/research`) ke Drive.


In [ ]:
# Colab bootstrap (Colab-only): mount drive + clone/pull repo + data/output links + kaggle token.
from pathlib import Path
import os
import shutil
import subprocess
import sys

COLAB_BOOTSTRAP_ENABLE = True
COLAB_REPO_URL = "https://github.com/akwancakra/nids-mscnn-bilstm-autoencoder.git"
COLAB_BRANCH = "master"
COLAB_REPO_DIR = Path("/content/nids-mscnn-bilstm-autoencoder")

# Configurable Drive Paths
COLAB_DRIVE_MOUNT = Path("/content/drive")
COLAB_DRIVE_ROOT = COLAB_DRIVE_MOUNT / "MyDrive"
# Project Folder in Drive (Persistent Storage)
# User specified path: /content/drive/MyDrive/nids-mscnn-bilstm-autoencoder/data/processed
COLAB_PROJECT_DRIVE_ROOT = COLAB_DRIVE_ROOT / "nids-mscnn-bilstm-autoencoder"

# Input Data (Raw)
COLAB_RAW_DATA_DRIVE = COLAB_DRIVE_ROOT / "nids-data" / "raw"

# Flags
COLAB_LINK_RAW_FROM_DRIVE = True
COLAB_LINK_OUTPUTS_TO_DRIVE = True
COLAB_FORCE_RELINK_OUTPUTS = True

def _is_colab_runtime() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

def _run_shell(cmd: list[str], cwd: Path | None = None) -> None:
    print("[CMD]", " ".join(cmd))
    subprocess.run(cmd, cwd=str(cwd) if cwd else None, check=True)

def _mount_drive() -> None:
    from google.colab import drive
    drive.mount(str(COLAB_DRIVE_MOUNT))

def _clone_or_pull_repo() -> None:
    if not COLAB_REPO_DIR.exists():
        _run_shell(["git", "clone", COLAB_REPO_URL, str(COLAB_REPO_DIR)], cwd=Path("/content"))
    else:
        _run_shell(["git", "fetch", "--all"], cwd=COLAB_REPO_DIR)
    
    try:
        _run_shell(["git", "checkout", COLAB_BRANCH], cwd=COLAB_REPO_DIR)
    except Exception:
        pass
        
    _run_shell(["git", "pull", "origin", COLAB_BRANCH], cwd=COLAB_REPO_DIR)

def _ensure_symlink_dir(repo_path: Path, drive_target: Path, allow_replace_existing: bool = False) -> None:
    """
    Robust symlink creation.
    1. Creates drive_target if not exists.
    2. Checks if repo_path is already a symlink to drive_target.
    3. If repo_path exists (dir/file) and allow_replace_existing is True, it removes it.
    4. Creates symlink.
    """
    drive_target.mkdir(parents=True, exist_ok=True)

    if repo_path.is_symlink():
        current_target = repo_path.resolve()
        if current_target == drive_target.resolve():
            print(f"[INFO] Symlink OK: {repo_path} -> {drive_target}")
            return
        repo_path.unlink(missing_ok=True)

    elif repo_path.exists():
        if allow_replace_existing:
            if repo_path.is_dir():
                shutil.rmtree(repo_path, ignore_errors=True)
            else:
                repo_path.unlink(missing_ok=True)
        else:
            # Fail-safe: do not silently delete real populated folders unless forced.
            try:
                has_contents = any(repo_path.iterdir())
            except Exception:
                has_contents = True
            
            if has_contents:
                 print(f"[WARN] Path exists and is not symlink: {repo_path}. Skipping link (not empty).")
                 return
            
            if repo_path.is_dir():
                repo_path.rmdir()
            else:
                repo_path.unlink(missing_ok=True)

    repo_path.parent.mkdir(parents=True, exist_ok=True)
    os.symlink(str(drive_target), str(repo_path), target_is_directory=True)
    print(f"[INFO] Symlink created: {repo_path} -> {drive_target}")

def _link_outputs_to_drive() -> None:
    # Map project folders to Drive folders
    # We want IntrusionDetectionSystem/data/processed to persist
    
    base_repo_project = COLAB_REPO_DIR / "IntrusionDetectionSystem"
    # User confirmed: processed data is in /content/drive/MyDrive/nids-mscnn-bilstm-autoencoder/data/processed
    # So base_drive_project should be the root of that path
    base_drive_project = COLAB_PROJECT_DRIVE_ROOT
    
    mapping = {
        # Persist Processed Data
        base_repo_project / "data" / "processed": base_drive_project / "data" / "processed",
        
        # Persist Models (Optional, but good for saving progress)
        base_repo_project / "models": base_drive_project / "models",
        
        # Persist Outputs/Logs
        base_repo_project / "output": base_drive_project / "output",
    }
    
    for repo_path, drive_target in mapping.items():
        _ensure_symlink_dir(
            repo_path,
            drive_target,
            allow_replace_existing=COLAB_FORCE_RELINK_OUTPUTS,
        )

IS_COLAB_RUNTIME = _is_colab_runtime()
print(f"IS_COLAB_RUNTIME={IS_COLAB_RUNTIME}")

if IS_COLAB_RUNTIME and COLAB_BOOTSTRAP_ENABLE:
    _mount_drive()
    _clone_or_pull_repo()
    
    if COLAB_LINK_OUTPUTS_TO_DRIVE:
        _link_outputs_to_drive()

    # Change working directory to project root
    project_sota_dir = COLAB_REPO_DIR / "IntrusionDetectionSystem"
    if project_sota_dir.exists():
        os.chdir(project_sota_dir)
        print(f"[INFO] Colab bootstrap done. cwd={Path.cwd()}")
    else:
        print(f"[WARN] Folder {project_sota_dir} tidak ditemukan.")
else:
    print("[SKIP] Colab bootstrap is disabled or not running in Colab.")

In [ ]:
# Resolve project root (works for local + Colab), lalu optional install dependency.
from pathlib import Path
import os
import subprocess
import sys

CANDIDATES = [
    Path.cwd(),
    Path.cwd().parent,
    Path("/content/nids-mscnn-bilstm-autoencoder/IntrusionDetectionSystem"), # Prioritaskan nama baru
    Path("/content/nids-cnn-lstm-autoencoder/IntrusionDetectionSystem"),     # Fallback nama lama
]

PROJECT_ROOT = None
for c in CANDIDATES:
    if (c / "Python").exists() and (c / "config.yaml").exists():
        PROJECT_ROOT = c.resolve()
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError("Project root tidak ditemukan. Pastikan notebook dijalankan dari folder IntrusionDetectionSystem.")

os.chdir(PROJECT_ROOT)
print(f"PROJECT_ROOT = {PROJECT_ROOT}")

# Add Python folder to sys.path
sys.path.append(os.path.join(PROJECT_ROOT, 'Python'))

INSTALL_DEPS = True
if INSTALL_DEPS:
    print("Installing dependencies...")
    # REMOVED: tensorflow-addons due to incompatibility with Python 3.10+
    subprocess.run([sys.executable, "-m", "pip", "install", "tensorflow", "pandas", "numpy", "matplotlib", "scikit-learn", "pyyaml", "joblib", "seaborn", "tqdm"], check=True)
    print("Dependencies installed.")
else:
    print("INSTALL_DEPS = False (skip install).")

In [ ]:
# @title Import Libraries & Modules
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import yaml
import glob
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import seaborn as sns
from tqdm.auto import tqdm
# import tensorflow_addons as tfa # Removed due to incompatibility

# Import modul custom yang sudah kita buat
from models.mscnn_bilstm_ae import build_multiscale_cnn_bilstm_ae
from preprocessing.data_loader_npz import load_npz_data
from utils.thresholding import calculate_reconstruction_error, get_threshold_percentile, plot_error_distribution

print("Modules imported successfully!")

## Data Preprocessing (New Pipeline)
Jalankan cell ini jika data processed belum tersedia. Script ini akan membaca RAW data dari Drive, melakukan cleaning, scaling (MinMax 0-1), dan sequencing, lalu menyimpannya ke folder processed project ini (yang tersimpan di Google Drive).

In [ ]:
# @title Run Data Preprocessing
RUN_PREPROCESSING = False # Set False jika data sudah ada

if RUN_PREPROCESSING and 'COLAB_RAW_DATA_ROOT' in locals():
    RAW_TRAIN_PATH = COLAB_RAW_DATA_ROOT / "CIC-IDS2017"
    RAW_TEST_PATH = COLAB_RAW_DATA_ROOT / "CSE-CIC-IDS2018"
    # OUTPUT KE FOLDER PROJECT UTAMA AGAR SYMLINK BEKERJA
    # Kita pastikan outputnya ke folder IntrusionDetectionSystem/data/processed
    OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
    
    print(f"Raw Train Path: {RAW_TRAIN_PATH}")
    print(f"Output Dir (Should be symlinked): {OUTPUT_DIR}")
    
    # Pastikan output dir ada
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    
    if RAW_TRAIN_PATH.exists():
        # Jalankan script preprocessing
        # Note: Proses ini bisa memakan waktu lama tergantung ukuran data
        !python Python/preprocessing/generate_sota_data.py \
            --raw_train "{RAW_TRAIN_PATH}" \
            --raw_test "{RAW_TEST_PATH}" \
            --output_dir "{OUTPUT_DIR}" \
            --seq_len 10 \
            --stride 1
    else:
        print("Raw data not found (check COLAB_RAW_DATA_ROOT). Skipping preprocessing.")
else:
    print("Skipping preprocessing step (RUN_PREPROCESSING=False or Not in Colab).")

In [ ]:
# @title QA Check: Verify Processed Data
# Jalankan cell ini untuk memastikan data hasil preprocessing sudah benar (Range 0-1, Tidak ada NaN)

def check_processed_data(data_path):
    if not data_path.exists():
        print(f"Directory not found: {data_path}")
        # Cek apakah symlink?
        if data_path.is_symlink():
            print(f"Symlink target: {data_path.resolve()}")
            if not data_path.resolve().exists():
                print("Symlink target BROKEN (not found).")
        return
        
    files = sorted(glob.glob(os.path.join(data_path, "*.npz")))
    if not files:
        print(f"No files found in {data_path}")
        return
        
    print(f"Checking random shard from: {data_path}")
    # Load random shard
    sample_file = files[0]
    data = np.load(sample_file)
    X = data['X']
    y = data['y']
    
    print(f"File: {os.path.basename(sample_file)}")
    print(f"Shape X: {X.shape}")
    print(f"Shape y: {y.shape}")
    print(f"Min Value: {np.min(X):.4f} (Should be >= 0.0)")
    print(f"Max Value: {np.max(X):.4f} (Should be <= 1.0)")
    print(f"Mean Value: {np.mean(X):.4f}")
    print(f"Has NaN: {np.isnan(X).any()}")
    print("-"*30)

if 'PROJECT_ROOT' in locals():
    processed_root = PROJECT_ROOT / "data" / "processed"
    print(f"Checking Processed Root: {processed_root}")
    
    if processed_root.exists():
        print("--- TRAIN DATA CHECK ---")
        check_processed_data(processed_root / "train")
        print("\n--- TEST DATA CHECK ---")
        check_processed_data(processed_root / "test")
    else:
        print(f"Processed data directory not found at: {processed_root}")
        if processed_root.is_symlink():
            print(f"It is a symlink pointing to: {processed_root.resolve()}")
            print("Check if your Google Drive path is correct and contains data.")
else:
    print("PROJECT_ROOT not defined. Run previous cells.")

## Konfigurasi Eksperimen
Kita definisikan parameter training dan path data di sini. 
**NOTE:** Kita menggunakan shards data yang sudah diproses dari project utama (`nids-cnn-lstm-autoencoder/data/research`) untuk konsistensi data.

In [ ]:
config = {
    'paths': {
        # Gunakan path RELATIF terhadap PROJECT_ROOT
        # Path ini akan menunjuk ke data hasil preprocessing baru
        'train_data': 'data/processed/train',
        'test_data': 'data/processed/test',
        'output_dir': 'output/mscnn_bilstm_ae_experiment',
        'model_save_path': 'models/mscnn_bilstm_ae_best.h5'
    },
    'model': {
        'sequence_length': 10,  # Sesuai data preprocessing (default script)
        'n_features': None,     # Akan dideteksi otomatis dari data
        'encoding_dim': 16,
        'learning_rate': 0.0005
    },
    'training': {
        'batch_size': 256,
        'epochs': 50,
        'validation_split': 0.1,
        'patience': 5
    },
    'thresholding': {
        'percentile': 99.9  # Sangat ketat untuk menghindari False Positives
    }
}

# Buat output directory
os.makedirs(config['paths']['output_dir'], exist_ok=True)
os.makedirs(os.path.dirname(config['paths']['model_save_path']), exist_ok=True)
print("Config loaded.")

## 1. Load Data Training (CIC-IDS2017)
Data ini harus berupa trafik **Benign (Normal)** saja.

In [ ]:
print("[Phase 1] Loading Training Data (NPZ Shards)...")
train_path = Path(config['paths']['train_data'])
if not train_path.exists():
    print(f"Warning: Path {train_path} not found.")

X_train, _ = load_npz_data(
    config['paths']['train_data'], 
    limit_files=None # Set angka (misal 10) jika ingin test cepat, None untuk semua data
)

if X_train is None:
    print("Error: No training data found! Check config path.")
else:
    print(f"Training Data Shape: {X_train.shape}")

## 2. Build & Train Model
Membangun model Multi-Scale CNN-BiLSTM Autoencoder dan melatihnya.
Kita gunakan `tqdm_callback` (jika tersedia) atau `verbose=1` Keras yang sudah memiliki progress bar bawaan yang bagus di Colab.

In [ ]:
if X_train is not None:
    print("[Phase 2] Building & Training Model...")

    # Split Validation
    X_train_split, X_val_split = train_test_split(
        X_train, 
        test_size=config['training']['validation_split'], 
        random_state=42
    )

    input_shape = (X_train.shape[1], X_train.shape[2])
    print(f"Input Shape: {input_shape}")

    # Build Model
    model = build_multiscale_cnn_bilstm_ae(input_shape, encoding_dim=config['model']['encoding_dim'])
    model.summary()

    # Callbacks
    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', 
            patience=config['training']['patience'], 
            restore_best_weights=True,
            verbose=1
        )
    ]
    
    # Try adding TQDM callback if available for smoother bar in some envs
    try:
        import tensorflow_addons as tfa
        callbacks.append(tfa.callbacks.TQDMProgressBar())
        verbose_mode = 0 # TQDM handles output
    except ImportError:
        verbose_mode = 1 # Default Keras progress bar (dynamic in Colab)

    # Train
    history = model.fit(
        X_train_split, X_train_split, # Autoencoder: Input == Target
        epochs=config['training']['epochs'],
        batch_size=config['training']['batch_size'],
        validation_data=(X_val_split, X_val_split),
        callbacks=callbacks,
        verbose=verbose_mode
    )

    # Save Model
    model.save(config['paths']['model_save_path'])
    print(f"Model saved to {config['paths']['model_save_path']}")

    # Plot Training History
    plt.figure(figsize=(10, 4))
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Val Loss')
    plt.title('Model Loss')
    plt.ylabel('Loss (MSE)')
    plt.xlabel('Epoch')
    plt.legend()
    plt.show()

## 3. Determine Threshold
Menentukan batas anomali berdasarkan distribusi error rekonstruksi pada data validasi (Benign).

In [ ]:
if X_train is not None:
    print("[Phase 3] Determining Threshold...")
    # calculate_reconstruction_error now uses TQDM
    val_errors = calculate_reconstruction_error(model, X_val_split)
    threshold = get_threshold_percentile(val_errors, percentile=config['thresholding']['percentile'])

    # Plot Distribution
    plot_error_distribution(
        val_errors, 
        threshold, 
        save_path=os.path.join(config['paths']['output_dir'], 'error_dist_val.png')
    )
    plt.show()

## 4. Evaluate on Test Data (CSE-CIC-IDS2018)
Menguji model pada data campuran (Benign + Attack) yang belum pernah dilihat sebelumnya.

In [ ]:
print("[Phase 4] Loading Test Data (NPZ Shards)...")
X_test, y_test = load_npz_data(
    config['paths']['test_data'], 
    limit_files=None # None = Load all test shards
)

if X_test is None:
    print("Error: No test data found!")
else:
    print(f"Test Data Shape: {X_test.shape}")
    print(f"Test Labels Shape: {y_test.shape}")
    
    if 'model' in locals():
        print("\n[Phase 5] Evaluation...")
        # TQDM enabled evaluation
        test_errors = calculate_reconstruction_error(model, X_test)
        
        # Predict: Error > Threshold -> Anomaly (1)
        y_pred = (test_errors > threshold).astype(int)
        
        # Calculate Metrics
        f1 = f1_score(y_test, y_pred)
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred)
        try:
            auc = roc_auc_score(y_test, test_errors)
        except:
            auc = 0.0
            
        cm = confusion_matrix(y_test, y_pred)
        
        print("\n" + "="*40)
        print("MSCNN-BiLSTM-AE EVALUATION REPORT")
        print("="*40)
        print(f"Threshold (Percentile {config['thresholding']['percentile']}): {threshold:.6f}")
        print(f"Accuracy:  {acc:.4f}")
        print(f"Precision: {prec:.4f}")
        print(f"Recall:    {rec:.4f}")
        print(f"F1-Score:  {f1:.4f}")
        print(f"AUC-ROC:   {auc:.4f}")
        print("-" * 20)
        print("Confusion Matrix:")
        print(cm)
        print("="*40)
        
        # Plot Confusion Matrix
        plt.figure(figsize=(6, 5))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Normal', 'Attack'], yticklabels=['Normal', 'Attack'])
        plt.ylabel('True Label')
        plt.xlabel('Predicted Label')
        plt.title('Confusion Matrix')
        plt.show()
    else:
        print("Model not found. Skipping evaluation.")

## 5. Experiment Summary
Menampilkan ringkasan hasil dari file `evaluation_results.yaml`.

In [ ]:
# @title Display Final Evaluation Results
import yaml
import pandas as pd
from IPython.display import display, Markdown

result_path = os.path.join(config['paths']['output_dir'], 'evaluation_results.yaml')

if os.path.exists(result_path):
    with open(result_path, 'r') as f:
        results = yaml.safe_load(f)
    
    print("\n" + "="*40)
    print("FINAL EXPERIMENT RESULTS")
    print("="*40)
    
    # Convert to DataFrame for nicer display
    metrics_df = pd.DataFrame([results])
    # Reorder columns if needed or just display
    display(metrics_df[['threshold', 'accuracy', 'precision', 'recall', 'f1_score', 'auc']])
    
    print("\nConfusion Matrix:")
    print(np.array(results['confusion_matrix']))
    
    # Optional: Display as Markdown table
    md_table = f"""
| Metric | Value |
| :--- | :--- |
| **Threshold** | `{results['threshold']:.6f}` |
| **Accuracy** | `{results['accuracy']:.4f}` |
| **Precision** | `{results['precision']:.4f}` |
| **Recall** | `{results['recall']:.4f}` |
| **F1-Score** | `{results['f1_score']:.4f}` |
| **AUC-ROC** | `{results['auc']:.4f}` |
"""
    display(Markdown(md_table))

else:
    print(f"Results file not found at {result_path}. Make sure the evaluation step completed successfully.")